In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Atenuarea erorilor prin rețele tensoriale (TEM): O Funcție Qiskit de la Algorithmiq
*Vezi [referința API](https://docs.quantum.ibm.com/api/functions/algorithmiq-tem)*

> **Note:** Funcțiile Qiskit sunt o funcționalitate experimentală disponibilă doar utilizatorilor din planul IBM Quantum&reg; Premium, Flex și On-Prem (prin API-ul IBM Quantum Platform). Acestea se află în stadiu de previzualizare și pot suferi modificări.


<Accordion>
<AccordionItem title="Versiuni pachete">

Codul de pe această pagină a fost dezvoltat folosind cerințele de mai jos.
Recomandăm folosirea acestor versiuni sau a unora mai noi.

```
qiskit[all]~=2.4.0
```
</AccordionItem>
</Accordion>
## Prezentare generală
Metoda TEM (Tensor-network Error Mitigation) de la Algorithmiq este un algoritm
hibrid cuantic-clasic conceput pentru atenuarea zgomotului exclusiv în etapa de
post-procesare clasică. Cu TEM, poți calcula valorile de așteptare ale
observabililor, atenuând erorile inevitabile induse de zgomot ce apar pe
hardware-ul cuantic, cu precizie sporită și eficiență a costurilor, ceea ce îl
face o opțiune extrem de atractivă atât pentru cercetătorii în domeniul cuantic,
cât și pentru practicienii din industrie.

Metoda constă în construirea unei rețele tensoriale care reprezintă inversul
canalului de zgomot global ce afectează starea procesorului cuantic și apoi în
aplicarea transformării asupra rezultatelor măsurătorilor complet informaționale
obținute din starea zgomotoasă, pentru a obține estimatori nebiasați ai
observabililor.

Ca avantaj, TEM valorifică măsurătorile complet informaționale pentru a oferi
acces la un set vast de valori de așteptare atenuate ale observabililor și are un
overhead de eșantionare optim pe hardware-ul cuantic, conform descrierii din
Filippov et al. (2023), [arXiv:2307.11740](https://arxiv.org/abs/2307.11740), și
Filippov et al. (2024), [arXiv:2403.13542](https://arxiv.org/abs/2403.13542).
Overhead-ul de măsurare se referă la numărul de măsurători suplimentare necesare
pentru atenuarea eficientă a erorilor, un factor critic în fezabilitatea
calculelor cuantice. Prin urmare, TEM are potențialul de a permite avantajul
cuantic în scenarii complexe, cum ar fi aplicații în domeniile haosului cuantic,
fizicii multi-corpuri, dinamicii Hubbard și simulărilor de chimie a moleculelor mici.

Principalele caracteristici și beneficii ale TEM pot fi rezumate astfel:

1.  **Overhead de măsurare optim**: TEM este optim față de
[limitele teoretice](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.131.210601),
ceea ce înseamnă că nicio metodă nu poate obține un overhead de măsurare mai
mic. Cu alte cuvinte, TEM necesită numărul minim de măsurători suplimentare
pentru atenuarea erorilor. Aceasta implică, la rândul său, că TEM utilizează
un timp de execuție cuantic minim.
2.  **Eficiență a costurilor**: Deoarece TEM gestionează atenuarea zgomotului
exclusiv în etapa de post-procesare, nu este nevoie să se adauge circuite
suplimentare calculatorului cuantic, ceea ce nu doar că face calculul mai
ieftin, ci și diminuează riscul introducerii unor erori suplimentare din
cauza imperfecțiunilor dispozitivelor cuantice.
3.  **Estimarea mai multor observabili**: Datorită măsurătorilor complet
informaționale, TEM estimează eficient mai mulți observabili cu aceleași
date de măsurare de la calculatorul cuantic.
4.  **Atenuarea erorilor de măsurare**: Funcția TEM Qiskit include și o
[metodă proprietară de atenuare a erorilor de citire](https://journals.aps.org/prresearch/abstract/10.1103/PhysRevResearch.5.033154)
capabilă să reducă semnificativ erorile de readout după o scurtă rulare de
calibrare.
5.  **Precizie**: TEM îmbunătățește semnificativ precizia și fiabilitatea

- [Request access to Algorithmiq Tensor-network error mitigation](https://quantum.ibm.com/functions?id=4b1b9d76-c18b-4788-b70b-15125111fbe6).
*See the [API reference](https://docs.quantum.ibm.com/api/functions/algorithmiq-tem)*
simulărilor cuantice digitale, făcând algoritmii cuantici mai exacți și mai
de încredere.
## Descriere
Funcția TEM îți permite să obții valori de așteptare atenuate ale erorilor
pentru mai mulți observabili pe un Circuit cuantic cu un overhead de eșantionare
minim. Circuitul este măsurat cu o măsură de operator cu valoare pozitivă
complet informațională (IC-POVM), iar rezultatele măsurătorilor colectate sunt
procesate pe un calculator clasic. Această măsurătoare este folosită pentru a
aplica metodele de rețele tensoriale și a construi o hartă de inversare a
zgomotului. Funcția aplică o transformare care inversează complet întregul circuit
zgomotos folosind rețele tensoriale pentru a reprezenta straturile zgomotoase.

![Schema TEM](../docs/images/guides/algorithmiq-tem/tem_scheme.svg "Estimarea atenuată a erorii unui observabil O prin post-procesarea rezultatelor de măsurare ale procesorului cuantic zgomotos. U și N denotă o operație cuantică ideală și harta de zgomot asociată, care poate fi în general nelocală (și extinsă la casetele gri). D reprezintă un tensor de operatori duali efectelor din măsurarea IC. Modulul de atenuare a zgomotului M este o rețea tensorială contractată eficient din mijloc spre exterior. Prima iterație a contracției este reprezentată de linia punctată violet, a doua de linia întreruptă, iar a treia de linia continuă.")

Odată ce circuitele sunt trimise funcției, acestea sunt transpilate și
optimizate pentru a minimiza numărul de straturi cu porți cu doi Qubiți (porțile
mai zgomotoase pe dispozitivele cuantice). Zgomotul care afectează straturile
este învățat prin
[Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/noise-learner-noise-learner)
folosind un model de zgomot sparse Pauli-Lindblad conform descrierii din E. van
den Berg, Z. Minev, A. Kandala, K. Temme, Nat. Phys. (2023).
[arXiv:2201.09866](https://arxiv.org/abs/2201.09866).

Modelul de zgomot este o descriere precisă a zgomotului de pe dispozitiv,
capabilă să capteze caracteristici subtile, inclusiv interferențele între Qubiți.
Totuși, zgomotul de pe dispozitive poate fluctua și deriva, iar zgomotul
învățat s-ar putea să nu fie precis la momentul la care se face estimarea. Acest
lucru poate duce la rezultate inexacte.
## Primii pași
Autentifică-te folosind [cheia API IBM Quantum Platform](http://quantum.cloud.ibm.com/) și selectează funcția TEM după cum urmează. (Acest fragment presupune că ți-ai [salvat deja contul](/guides/functions#install-qiskit-functions-catalog-client) în mediul local.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

tem_function_name = "algorithmiq/tem"
catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Load your function
tem = catalog.load(tem_function_name)

## Exemplu
Fragmentul următor arată un exemplu în care TEM este utilizat pentru a calcula valorile de așteptare ale unui observabil dat un Circuit cuantic simplu.

In [2]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

# Create a quantum circuit
qc = QuantumCircuit(3)
qc.u(0.4, 0.9, -0.3, 0)
qc.u(-0.4, 0.2, 1.3, 1)
qc.u(-1.2, -1.2, 0.3, 2)
for _ in range(2):
    qc.barrier()
    qc.cx(0, 1)
    qc.cx(2, 1)
    qc.barrier()
    qc.u(0.4, 0.9, -0.3, 0)
    qc.u(-0.4, 0.2, 1.3, 1)
    qc.u(-1.2, -1.2, 0.3, 2)

# Define the observables
observable = SparsePauliOp("IYX", 1.0)

# Define the execution options
pub = (qc, [observable])
options = {"default_precision": 0.02}

# Define backend to use. TEM will choose the least-busy device
# reported by IBM if not specified
backend_name = "ibm_marrakesh"

# Run the TEM function (uses around three minutes of QPU time)
job = tem.run(pubs=[pub], backend_name=backend_name, options=options)

Folosește API-urile Qiskit Serverless pentru a verifica statusul sarcinii tale Qiskit Function:

In [3]:
print(job.status())

QUEUED


You can return the results as:

In [4]:
result = job.result()
evs = result[0].data.evs
print(evs[0])

0.02165380888171687


Poți obține rezultatele astfel: